# Getting Started
* Create a new API key at https://www.console.anthropic.com
* Create a .env file
  * Add ```ANTHROPIC_API_KEY="<your_api_key>"```
* Ensure uv is installed: https://docs.astral.sh/uv/getting-started/installation/
* Run Standalone Notebook:
  * > uv run jupyter lab
* Run Notebook in VSCode:
  * Open Jupyter Notebook and select `.venv (Python 3.12.11)` as the kernel

In [ ]:
# environment setup
from dotenv import load_dotenv
from datetime import datetime, timedelta
import json
load_dotenv()

from anthropic import Anthropic
from anthropic.types import ToolParam, Message
client = Anthropic()
model = 'claude-haiku-4-5'
max_tokens = 1000

In [55]:
# helper functions
def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)

def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None, temperature=1.0, tools=[]):
  params = {
    'model': model,
    'max_tokens': max_tokens,
    'messages': messages,
    'temperature': temperature,
  }

  if tools:
    params['tools'] = tools

  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)

  return message

def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[]
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)

def text_from_message(message):
  return "\n".join([
    block.text for block in message.content if block.type == "text"
  ])

## What are Tool Functions?
A tool function is a plain Python function that gets executed automatically when Claude decides it needs extra information tl help a user. For example, if someone asks "what's the weather?" Claude would call your weather tool to get the current weather.
**Best practices**
* Use well-named, descriptive arguemnts
* Validate the inputs, raising an error if they fail validation
* Return meaningful errors, Claude will try to call to use your function a second time

In [37]:
# Tool 1
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    
    return datetime.now().strftime(date_format)

# Tool 2
def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

# Tool 3
def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")



# JSON schemas
Json schema isn't specific to AI or tool calling, its widely used data validation specification. The AI community adapted it because it's convenient to describe function parameters and validate data.
### Best practices
* Explain what the tool does, when to use it, and what it returns
* Aim for 3 to 4 sentences
* Provide super detailed descriptions

Instead of writing JSON schemas from scratch, you can use Claude itself to generate them. Here's the process:
1. Copy your tool function code
2. Go to Claude and ask it to write a JSON schema for tool calling
3. Include the Anthropic documentation on tool use as context
4. Let Claude generate a properly formatted schema following best practices.

The prompt should be something like "Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation"

Tool use link: https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview

In [38]:
# Use ToolParam for better type checking
get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current local date and time as a formatted string. Use this tool whenever the user asks what the current date, time, day of week, or timestamp is, or when a task requires knowing 'now' (e.g. timestamping a file, calculating a relative date, logging an event). The output format is controlled by the date_format parameter, which accepts a Python strftime-style format string. If omitted, the default format '%Y-%m-%d %H:%M:%S' is used (e.g. '2026-07-15 14:30:00').",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A strftime-compatible format string describing how to render the date/time. Common examples: '%Y-%m-%d' -> '2026-07-15'; '%H:%M:%S' -> '14:30:00'; '%A, %B %d, %Y' -> 'Wednesday, July 15, 2026'. Must not be an empty string. If not provided, defaults to '%Y-%m-%d %H:%M:%S'.",
        "minLength": 1,
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

add_duration_to_datetime_schema = ToolParam({
  "name": "add_duration_to_datetime",
  "description": "Adds or subtracts a duration of time from a given datetime and returns the resulting date/time as a formatted string. Use this tool when the user asks for a date or time relative to a known starting point — for example, 'what date is 45 days after 2024-01-15?' or 'what time is 3 hours before 2024-06-01 14:00:00?'. Do not use this tool to parse vague or relative natural-language dates (e.g. 'next Tuesday', 'a couple weeks ago') — 'datetime_str' must already be an exact string matching 'input_format'. This tool has no time zone awareness, so it should not be used for time zone conversions. To subtract time instead of adding it, pass a negative value for 'duration'. When 'unit' is 'months' or 'years', calendar-aware arithmetic is used rather than a fixed number of days — for example, adding 1 month to January 31 lands on February 28 or 29, not March 3. The output is always returned as a string in the fixed format 'Weekday, Month DD, YYYY HH:MM:SS AM/PM' (e.g. 'Monday, January 15, 2024 02:30:00 PM'), regardless of what 'input_format' was.",
  "input_schema": {
    "type": "object",
    "properties": {
      "datetime_str": {
        "type": "string",
        "description": "The starting date/time to adjust, given as a string that exactly matches the pattern specified in 'input_format'. For example, if 'input_format' is the default '%Y-%m-%d', this should look like '2024-01-15'."
      },
      "duration": {
        "type": "integer",
        "description": "The amount of time to add to 'datetime_str', measured in the unit given by 'unit'. Use a negative integer to subtract time instead (e.g. -7 with unit 'days' goes 7 days into the past). Defaults to 0, which returns the input datetime reformatted but otherwise unchanged.",
        "default": 0
      },
      "unit": {
        "type": "string",
        "description": "The unit of time that 'duration' is measured in. 'months' and 'years' use calendar-aware arithmetic (accounting for varying month lengths and leap years) rather than a fixed-length approximation; all other units are exact fixed-length durations. Defaults to 'days'.",
        "enum": ["seconds", "minutes", "hours", "days", "weeks", "months", "years"],
        "default": "days"
      },
      "input_format": {
        "type": "string",
        "description": "A Python strptime-style format string describing the exact layout of 'datetime_str' (e.g. '%Y-%m-%d' for '2024-01-15', or '%m/%d/%Y %H:%M' for '01/15/2024 14:30'). Parsing will fail if 'datetime_str' does not match this format exactly. Defaults to '%Y-%m-%d'.",
        "default": "%Y-%m-%d"
      }
    },
    "required": ["datetime_str"]
  }
})

set_reminder_schema = ToolParam({
  "name": "set_reminder",
  "description": "Records a reminder by printing its content and target time to the console/log. Use this tool when the user explicitly asks to be reminded of something at a specific time (e.g. 'remind me to call the vet tomorrow at 9am'). Important limitation: this tool does not schedule any actual notification, alarm, or calendar event, and the reminder is not stored anywhere durable — it only writes the reminder to output for logging or debugging purposes. Do not imply to the user that they will receive a notification or alert at the specified time as a result of calling this tool. The 'timestamp' parameter is not parsed or validated by the function, so pass it as a clear, human-readable string rather than assuming any particular format is enforced.",
  "input_schema": {
    "type": "object",
    "properties": {
      "content": {
        "type": "string",
        "description": "The text of the reminder — what the user wants to be reminded about (e.g. 'Call the vet about Max's checkup'). Should be a concise, self-contained description of the task or event."
      },
      "timestamp": {
        "type": "string",
        "description": "The date and/or time the reminder is for, as a human-readable string (e.g. '2024-06-01 09:00' or 'tomorrow at 9am'). This is not parsed or validated by the tool, so prefer an unambiguous, explicit format such as ISO 8601 ('YYYY-MM-DD HH:MM') when the exact time matters."
      }
    },
    "required": ["content", "timestamp"]
  }
})

In [10]:
messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")

response = chat(messages)

print(response)

I don't have access to real-time information, so I can't tell you the exact current time. 

To find the current time, you can:
- Check your device's clock
- Search "current time" online
- Ask a voice assistant
- Visit a time website like timeanddate.com


In [16]:
messages = []
add_user_message(messages, "what is the exact time, formatted as HH:MM:SS?")

# Notice that you will get a message block with two messages
# * "Text" Block - human readable text explaining what Claude is doing (like I can help you find the current time)
# * "ToolUse" Block - Instructions for your code about which tool to call and what parameters to use
response = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=messages,
    tools=[get_current_datetime_schema]
)
print(response)

messages.append({
    "role": "assistant",
    "content": response.content
})

print(messages)


Message(id='msg_011Cd31b5P4WfBC5NzJaYDiJ', container=None, content=[ToolUseBlock(id='toolu_0184dDwo41szwLRK3ZghcUf1', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=824, output_tokens=63, server_tool_use=None, service_tier='standard'), stop_details=None)
[{'role': 'user', 'content': 'what is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_0184dDwo41szwLRK3ZghcUf1', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


In [22]:
result = get_current_datetime(**response.content[0].input)

In [ ]:
# We have to append a user message with the result from the tool call
# Make sure to include the tool use id in the content. This is very important
# as it is used to differentiate results from multiple tool calls
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": result,
            "is_error": False,
        }
    ]
})

print(messages)

[{'role': 'user', 'content': 'what is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_0184dDwo41szwLRK3ZghcUf1', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_0184dDwo41szwLRK3ZghcUf1', 'content': '19:03:46', 'is_error': False}]}]


In [25]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_011Cd32LJYXDPyBqT6uiBo5y', container=None, content=[TextBlock(citations=None, text='The exact time is **19:03:46** (7:03:46 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=904, output_tokens=23, server_tool_use=None, service_tier='standard'), stop_details=None)

In [81]:
def run_tool(tool_name, tool_input):
    match tool_name:
        case "get_current_datetime":
            return get_current_datetime(**tool_input)
        case "add_duration_to_datetime":
            return add_duration_to_datetime(**tool_input)
        case "set_reminder":
            return set_reminder(**tool_input)

def run_tools(message):
    # Find blocks that are specific to tool uses
    tool_resquests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []

    # make a tool call for each open request
    for tool_request in tool_resquests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }
        tool_result_blocks.append(tool_result_block)
    
    return tool_result_blocks

In [ ]:
def run_conversation(messages, tools=[]):
    while True:
        response = chat(messages, tools=tools)

        add_assistant_message(messages, response)
        print(text_from_message(response))

        # Check whether Claude wants to use a tool.
        # If it doesn't we need to break out of loop
        if response.stop_reason != 'tool_use':
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [83]:
messages = []
# Claude will likely make two calls
add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?"
)

res = run_conversation(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])

print(res)



The current time is:
- **HH:MM format**: 16:49
- **SS format**: 44

So the current time is 16:49:44 (4:49:44 PM).
[{'role': 'user', 'content': [{'type': 'text', 'text': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'}]}, {'role': 'assistant', 'content': [{'type': 'tool_use', 'id': 'toolu_01ChFiwGv6CRrrEHnrMbjnnj', 'name': 'get_current_datetime', 'input': {'date_format': '%H:%M'}}, {'type': 'tool_use', 'id': 'toolu_011oFNq4GmnDdzKJeKNoBja2', 'name': 'get_current_datetime', 'input': {'date_format': '%S'}}]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01ChFiwGv6CRrrEHnrMbjnnj', 'content': '"16:49"', 'is_error': False}, {'type': 'tool_result', 'tool_use_id': 'toolu_011oFNq4GmnDdzKJeKNoBja2', 'content': '"44"', 'is_error': False}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'The current time is:\n- **HH:MM format**: 16:49\n- **SS format**: 44\n\nSo the current time is 16:49:44 (4:49:44 PM).'}]}]


In [ ]:
# Multi tool calls
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050."
)

res = run_conversation(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])
print(res)

I need to calculate what date is 177 days after January 1st, 2050.
Now I'll set a reminder for your doctor's appointment on that date:
----
Setting the following reminder for Monday, June 27, 2050:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** (which is 177 days after January 1st, 2050).
[{'role': 'user', 'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='I need to calculate what date is 177 days after January 1st, 2050.', type='text'), ToolUseBlock(id='toolu_01WEGKvPNdMqY2nE1MkNohKT', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01WEGKvPNdMqY2nE1MkNohKT', 'content': '"Monday, June 27, 2050 12:00:00 AM"', 'is_error': False}]}, 

# Streaming Results

In [61]:
def save_article(**kwargs):
    return "Article saved!"

save_article_schema = ToolParam({
    "name": "save_article",
    "description": "Saves a scholarly journal article",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "Abstract of the article. One short sentence max",
            },
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {
                        "type": "integer",
                        "description": "Word count",
                    },
                    "review": {
                        "type": "string",
                        "description": "Eight sentence review of the paper",
                    },
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
    })
save_short_article_schema = ToolParam({
    "name": "save_article",
    "description": "Saves a scholarly journal article",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "Abstract of the article. One short sentence max",
            },
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {
                        "type": "integer",
                        "description": "Word count",
                    },
                    "review": {
                        "type": "string",
                        "description": "Review of paper. One short sentence max",
                    },
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
    })

In [73]:
def run_conversation_stream(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [63]:
messages = []
add_user_message(
    messages,
    "Create and save a fake biology article",
)

run_conversation_stream(
    messages,
    tools=[save_article_schema],
    fine_grained=True # This enables true streaming, but needs to be better at handling errors as it would return invalid json
)

I'll create and save a fake biology article for you.


>>> Tool Call: "save_article"
{"abstract": "This study investigates the novel metabolic pathways of bioluminescent fungi and their potential applications in sustainable energy production.", "meta": {
  "word_count": 4250,
  "review": "This paper presents an interesting exploration of bioluminescent fungal metabolism and its practical applications in renewable energy. The authors conduct a comprehensive review of existing literature on fungal bioluminescence mechanisms and propose several novel pathways that could be exploited for energy generation. The experimental design is well-structured with appropriate controls and methodologies that follow standard laboratory protocols. However, the actual experimental data is somewhat limited, relying heavily on preliminary observations rather than extensive empirical validation. The authors make ambitious claims about commercial viability that are not fully supported by the evidence present

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake biology article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake biology article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01Nr3f1xWnbATFDrhWYj9ouT',
    'name': 'save_article',
    'input': {'abstract': 'This study investigates the novel metabolic pathways of bioluminescent fungi and their potential applications in sustainable energy production.',
     'meta': {'word_count': 4250,
      'review': 'This paper presents an interesting exploration of bioluminescent fungal metabolism and its practical applications in renewable energy. The authors conduct a comprehensive review of existing literature on fungal bioluminescence mechanisms and propose several novel pathways that could be exploited for energy generation. The experimental design is well-structured with appropriate controls and methodologies that follow standard laboratory protocols. H